# 06 ANNOVAR input formatting

Convert each selected run's full notebook-04 per-variant table into ANNOVAR's five-field input format ["Chr", "Start", "End", "REF", "ALT"] while retaining all original variant and annotation columns.

This notebook only creates `.avinput` files. ANNOVAR output reconstruction and downstream filtering are performed in notebook 07.

In [ ]:
from pathlib import Path
import pandas as pd

notebook_dir = Path("/home/donetski/Notebooks")
input_dir = notebook_dir / "OutputFiles" / "04_qc_checking_on_target"
output_dir = notebook_dir / "OutputFiles" / "06_annovar_input"

output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
output_prefix = "06"
runs_to_process = "all"  # "all", "Run2", or ["Run2", "Run4"]

all_runs = ["Run1", "Run2", "Run3", "Run4"]

if isinstance(runs_to_process, str):
    runs = all_runs if runs_to_process.lower() == "all" else [runs_to_process]
else:
    runs = runs_to_process

chr_col = "Chr"
start_col = "Start"
ref_col = "REF"
alt_col = "ALT"

required_cols = [chr_col, start_col, ref_col, alt_col]
print("Runs to process:", runs)

## Define the ANNOVAR allele conversion

ANNOVAR requires the first five fields:

```text
Chr    Start    End    Ref    Alt

In [ ]:
def clean_allele(value):
    if pd.isna(value):
        return "-"
    value = str(value).strip().upper()
    return "-" if value in {"", ".", "NAN", "NONE", "NULL"} else value


def shared_prefix_length(ref, alt):
    i = 0
    while i < min(len(ref), len(alt)) and ref[i] == alt[i]:
        i += 1
    return i


def convert_variant(row):
    chrom, pos = row[chr_col], int(row[start_col])
    ref, alt = clean_allele(row[ref_col]), clean_allele(row[alt_col])

    if ref == "-" and alt != "-":
        return pd.Series([chrom, pos, pos, "-", alt])
    if alt == "-" and ref != "-":
        return pd.Series([chrom, pos, pos + len(ref) - 1, ref, "-"])
    if len(ref) == 1 and len(alt) == 1:
        return pd.Series([chrom, pos, pos, ref, alt])
    if len(ref) == len(alt) and ref != alt:
        return pd.Series([chrom, pos, pos + len(ref) - 1, ref, alt])

    prefix = shared_prefix_length(ref, alt)
    ref_trim, alt_trim = ref[prefix:], alt[prefix:]

    if not ref_trim and alt_trim:
        return pd.Series([chrom, pos + prefix - 1, pos + prefix - 1, "-", alt_trim])
    if ref_trim and not alt_trim:
        return pd.Series([chrom, pos + prefix, pos + prefix + len(ref_trim) - 1, ref_trim, "-"])
    if ref_trim and alt_trim:
        return pd.Series([chrom, pos + prefix, pos + prefix + len(ref_trim) - 1, ref_trim, alt_trim])

    return pd.Series([chrom, pos, pos, ref, alt])

## Process each selected run

Each run is loaded and processed independently.

The resulting table contains:

1. the five normalized ANNOVAR fields;
2. all original notebook-04 columns in their original order.

No rows are deduplicated or filtered.

In [ ]:
def process_run(run):
    input_file = input_dir / f"04_{run}_per_variant_target_status_FULL.csv"
    df = pd.read_csv(input_file, low_memory=False)
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{run} is missing columns: {missing_cols}")

    df = df.copy()
    df[chr_col] = (df[chr_col].astype("string").str.strip().str.replace(r"^chr", "", regex=True, case=False))
    df[start_col] = pd.to_numeric(df[start_col], errors="raise").astype("int64")
    df[ref_col] = df[ref_col].map(clean_allele)
    df[alt_col] = df[alt_col].map(clean_allele)

    annovar_fields = df.apply(convert_variant, axis=1)
    annovar_fields.columns = ["Chr", "Start", "End", "Ref", "Alt"]
    annovar_input = pd.concat([annovar_fields, df.reset_index(drop=True)],axis=1,)

    print(f"{run}: {len(df):,} input rows | {len(annovar_input):,} ANNOVAR rows")
    return annovar_input

## Save the ANNOVAR input files

Each result is written as a tab-delimited `.avinput` file.

The header begins with `#`, preserving the header behavior used by the existing ANNOVAR workflow. Because the current ANNOVAR run preserves these original column names in its output, notebook 07 will not need to reconstruct them from an embedded `Otherinfo1` header row.

In [ ]:
def save_run_output(run, annovar_input):
    output_file = output_dir / f"{output_prefix}_{run}_annovar_input.avinput"

    with output_file.open("w", encoding="utf-8", newline="") as f:
        f.write("#" + "\t".join(annovar_input.columns) + "\n")
        annovar_input.to_csv(f, sep="\t", index=False, header=False,)
    print(f"Saved: {output_file}")
    return output_file

## Validate the written ANNOVAR file

The validation confirms that:

- the saved file has the same number of rows as the processed dataframe;
- the first five ANNOVAR fields are present;
- Start and End are numeric;
- no variant has End before Start;
- the number of insertions and deletions can be reviewed.

This validation does not remove or modify any rows.

In [ ]:
def validate_run_output(run, output_file, expected_rows):
    saved = pd.read_csv(
        output_file,
        sep="\t",
        dtype=str,
        low_memory=False,
    )

    first_five = saved.iloc[:, :5].copy()
    first_five.columns = ["Chr", "Start", "End", "Ref", "Alt"]

    start = pd.to_numeric(first_five["Start"], errors="coerce")
    end = pd.to_numeric(first_five["End"], errors="coerce")

    validation = pd.DataFrame([{
        "run": run,
        "expected_rows": expected_rows,
        "saved_rows": len(saved),
        "same_row_count": expected_rows == len(saved),
        "missing_first_five_values": int(first_five.isna().sum().sum()),
        "invalid_start_or_end": int((start.isna() | end.isna()).sum()),
        "end_before_start": int((end < start).fillna(False).sum()),
        "insertions": int(first_five["Ref"].eq("-").sum()),
        "deletions": int(first_five["Alt"].eq("-").sum()),
        "output_file": str(output_file),
    }])

    return validation

## Run the formatting workflow

Process each selected run separately, save its ANNOVAR input, and collect a compact validation summary.

In [ ]:
validation_results = []

for run in runs:
    annovar_input = process_run(run)
    output_file = save_run_output(run, annovar_input)

    validation_results.append(
        validate_run_output(
            run=run,
            output_file=output_file,
            expected_rows=len(annovar_input),
        )
    )

validation_summary = pd.concat(validation_results, ignore_index=True)
validation_summary

In [ ]:
for run in runs:
    run_validation = validation_summary[validation_summary["run"] == run]
    qc_file = output_dir / f"{output_prefix}_{run}_annovar_input_sanity_check.csv"
    run_validation.to_csv(qc_file, index=False)
    print(f"Saved sanity check: {qc_file}")